In [ ]:
# Install unsloth first so it resolves its own dependency pins,
# then install remaining packages within unsloth's supported ranges.
# Restart runtime when prompted, then skip this cell.
!pip install unsloth -q
!pip install "trl<=0.24.0" "transformers<=5.5.0" "datasets>=3.4.1,<4.4.0" \
    accelerate peft bitsandbytes scikit-learn -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update these paths if you stored files elsewhere in Drive
JSONL_PATH        = '/content/drive/MyDrive/TruthBite/synthetic_cot_dataset.jsonl'
EU_ADDITIVES_PATH = '/content/drive/MyDrive/TruthBite/eu_food_additives.json'
SAVE_DIR          = '/content/drive/MyDrive/TruthBite/model'

MODEL_NAME    = 'unsloth/phi-4-mini-instruct'
MAX_SEQ_LEN   = 2048
LORA_RANK     = 16
LEARNING_RATE = 2e-4
BATCH_SIZE    = 2
GRAD_ACCUM    = 4
EPOCHS        = 3
SEED          = 42
TEST_SIZE     = 0.2


In [ ]:
import json, re

ENUM_RE = re.compile(r'\bE\d{3,4}[A-Z]?\b', re.IGNORECASE)

SYSTEM_PROMPT = (
    'You are a senior Food Scientist specialized in NOVA food processing classification.\n'
    'Return STRICT JSON (no markdown) with these exact keys:\n'
    '  ingredient_steps: list of objects each with: ingredient, analysis, nova_marker, e_number, cited_function\n'
    '  reasoning_summary: string\n'
    '  predicted_nova_group: integer 1-4\n'
    'Rules:\n'
    '1) Analyse each ingredient step-by-step.\n'
    '2) When an additive appears, cite its E-number and function from the EU additive context.\n'
    '3) Be factually grounded and concise.\n'
    '4) Output only valid JSON, no markdown fences.'
)

def extract_text(raw):
    if not raw:
        return ''
    if isinstance(raw, dict):
        return raw.get('text', '')
    s = str(raw).strip()
    if not s.startswith('[{'):
        return s
    # Ingredient text stored as stringified list-of-dicts from OFF parquet
    key = "'text': '"
    i = s.find(key)
    if i == -1:
        return s
    start = i + len(key)
    end = s.find("'", start)
    return s[start:end] if end != -1 else s

def build_user_message(record, eu_additives):
    ingredients = extract_text(record.get('ingredients_text', ''))
    product     = extract_text(record.get('product_name', 'Unknown')) or 'Unknown'
    country     = record.get('country') or 'Unknown'
    ing_lower   = ingredients.lower()

    ctx = {}

    # Match explicit E-number codes written in the text (e.g. E322)
    for code in ENUM_RE.findall(ingredients.replace('-', '')):
        code = code.upper()
        if code in eu_additives:
            ctx[code] = eu_additives[code]

    # Match by aliases (e.g. 'Lecithin' in E322 aliases matches 'Soya Lecithin')
    for code, entry in eu_additives.items():
        if code in ctx:
            continue
        for alias in entry.get('aliases', []):
            if len(alias) > 4 and alias.lower() in ing_lower:
                ctx[code] = entry
                break

    lines = []
    for code, entry in sorted(ctx.items()):
        name  = entry.get('name', '')
        funcs = [fn.replace('en:', '') for fn in entry.get('functions', [])]
        lines.append(f'{code}: name={name}, functions={funcs}')

    block = '\n'.join(lines) if lines else 'No explicit E-number found.'
    return (
        f'Product: {product}\nCountry: {country}\n'
        f'Ingredients: {ingredients}\nEU additive context:\n{block}'
    )


In [ ]:
with open(EU_ADDITIVES_PATH) as f:
    eu_additives = json.load(f)

records = []
with open(JSONL_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'Loaded {len(records)} traces')


In [ ]:
from collections import Counter

dist = Counter(r['ground_truth_nova_group'] for r in records)
for g in sorted(dist):
    print(f'  NOVA {g}: {dist[g]} ({dist[g]/len(records)*100:.1f}%)')

print('\n--- Sample user message ---')
print(build_user_message(records[0], eu_additives))
print('\n--- Sample assistant response (first 500 chars) ---')
print(json.dumps(records[0]['trace'], indent=2)[:500])


In [ ]:
from sklearn.model_selection import train_test_split

def make_conversation(record):
    return {
        'messages': [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': build_user_message(record, eu_additives)},
            {'role': 'assistant', 'content': json.dumps(record['trace'], ensure_ascii=False)},
        ],
        'nova_group': record['ground_truth_nova_group'],
    }

conversations = [make_conversation(r) for r in records]
labels = [c['nova_group'] for c in conversations]

train_data, test_data = train_test_split(
    conversations, test_size=TEST_SIZE, random_state=SEED, stratify=labels
)

print(f'Train: {len(train_data)} | Test: {len(test_data)}')
print('Train NOVA:', dict(sorted(Counter(d['nova_group'] for d in train_data).items())))
print('Test  NOVA:', dict(sorted(Counter(d['nova_group'] for d in test_data).items())))


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
model.print_trainable_parameters()


In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

# Phi-4 and Phi-3 share the same chat template format
tokenizer = get_chat_template(tokenizer, chat_template='phi-3')

def apply_template(examples):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in examples['messages']
    ]
    return {'text': texts}

train_ds = Dataset.from_list([{'messages': d['messages']} for d in train_data])
train_ds = train_ds.map(apply_template, batched=True)

print(f'Training examples: {len(train_ds)}')
print('\nFormatted sample (first 500 chars):')
print(train_ds[0]['text'])


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
import os

os.makedirs(f'{SAVE_DIR}/checkpoints', exist_ok=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.1,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=SEED,
        output_dir=f'{SAVE_DIR}/checkpoints',
        save_strategy='epoch',
        report_to='none',
    ),
)

# Auto-resume from the latest epoch checkpoint if the session was interrupted
ckpt_dir = f'{SAVE_DIR}/checkpoints'
checkpoints = [d for d in os.listdir(ckpt_dir) if d.startswith('checkpoint-')]
resume = ckpt_dir if checkpoints else False
if resume:
    print(f'Resuming from checkpoint: {sorted(checkpoints)[-1]}')

stats = trainer.train(resume_from_checkpoint=resume)
print(f'Training complete. Loss: {stats.training_loss:.4f}')

In [ ]:
# ~30-60 min on T4 for 341 examples — can run overnight
import torch
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report

FastLanguageModel.for_inference(model)

def predict(conv):
    ids = tokenizer.apply_chat_template(
        conv['messages'][:2],
        tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=1024, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    try:
        return int(json.loads(text).get('predicted_nova_group', -1))
    except Exception:
        idx = text.find('predicted_nova_group')
        if idx != -1:
            chunk = text[idx:idx + 30]
            for n in (1, 2, 3, 4):
                if str(n) in chunk:
                    return n
        return -1

y_true, y_pred = [], []
for conv in tqdm(test_data, desc='Evaluating'):
    y_true.append(conv['nova_group'])
    y_pred.append(predict(conv))

failures = sum(1 for p in y_pred if p == -1)
valid = [(t, p) for t, p in zip(y_true, y_pred) if p != -1]
yt, yp = zip(*valid) if valid else ([], [])

print(f'Format failures: {failures}/{len(test_data)}')
if yt:
    acc = accuracy_score(yt, yp)
    f1  = f1_score(yt, yp, average='macro', labels=[1, 2, 3, 4], zero_division=0)
    print(f'NOVA Accuracy: {acc:.4f}')
    print(f'Macro F1:      {f1:.4f}')
    print(classification_report(
        yt, yp, labels=[1, 2, 3, 4],
        target_names=['NOVA 1', 'NOVA 2', 'NOVA 3', 'NOVA 4'],
        zero_division=0
    ))


In [ ]:
import os

adapter_path = f'{SAVE_DIR}/lora_adapter'
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'Adapter saved to {adapter_path}')

gguf_dir  = f'{SAVE_DIR}/gguf'
gguf_name = f'{gguf_dir}/truthbite-phi4'
os.makedirs(gguf_dir, exist_ok=True)
model.save_pretrained_gguf(gguf_name, tokenizer, quantization_method='q4_k_m')
print(f'GGUF saved to {gguf_name}.Q4_K_M.gguf')

modelfile = (
    f'FROM {gguf_name}.Q4_K_M.gguf\n\n'
    f'SYSTEM """\n{SYSTEM_PROMPT}\n"""\n\n'
    'TEMPLATE """\n'
    '{{ if .System }}<|system|>\n{{ .System }}<|end|>\n{{ end }}'
    '{{ range .Messages }}{{ if eq .Role "user" }}<|user|>\n{{ .Content }}<|end|>\n'
    '{{ else if eq .Role "assistant" }}<|assistant|>\n{{ .Content }}<|end|>\n'
    '{{ end }}{{ end }}<|assistant|>\n"""\n\n'
    'PARAMETER temperature 0.1\n'
    'PARAMETER num_ctx 4096\n'
    'PARAMETER stop "<|end|>"\n'
)
mf_path = f'{gguf_dir}/Modelfile'
with open(mf_path, 'w') as fh:
    fh.write(modelfile)
print(f'Modelfile written to {mf_path}')
print('\nTo load into Ollama after downloading the GGUF and Modelfile:')
print('  ollama create truthbite-phi4 -f Modelfile')
print('  ollama list')

In [ ]:
import subprocess

subprocess.run(['git', 'clone', '--depth=1',
                'https://github.com/ggerganov/llama.cpp',
                '/tmp/llama_cpp_latest'], check=True)
subprocess.run(['pip', 'install', '-r', '/tmp/llama_cpp_latest/requirements.txt', '-q'], check=True)

result = subprocess.run([
    '/usr/bin/python3', '/tmp/llama_cpp_latest/convert_hf_to_gguf.py',
    f'{SAVE_DIR}/gguf/truthbite-phi4',
    '--outfile', '/tmp/truthbite-phi4.F16.gguf',
    '--outtype', 'f16',
], check=False, capture_output=True, text=True)
print(result.stderr[-2000:])
print('Return code:', result.returncode)

In [ ]:
subprocess.run([
    '/root/.unsloth/llama.cpp/llama-quantize',
    '/tmp/truthbite-phi4.F16.gguf',
    '/tmp/truthbite-phi4.Q4_K_M.gguf',
    'q4_k_m',
], check=True)
print('Quantization done.')

In [ ]:
import shutil

shutil.copy('/tmp/truthbite-phi4.Q4_K_M.gguf',
            f'{SAVE_DIR}/gguf/truthbite-phi4.Q4_K_M.gguf')
print('Copied to Drive.')

modelfile = (
    'FROM ./truthbite-phi4.Q4_K_M.gguf\n\n'
    f'SYSTEM """\n{SYSTEM_PROMPT}\n"""\n\n'
    'TEMPLATE """\n'
    '{{ if .System }}<|system|>\n{{ .System }}<|end|>\n{{ end }}'
    '{{ range .Messages }}{{ if eq .Role "user" }}<|user|>\n{{ .Content }}<|end|>\n'
    '{{ else if eq .Role "assistant" }}<|assistant|>\n{{ .Content }}<|end|>\n'
    '{{ end }}{{ end }}<|assistant|>\n"""\n\n'
    'PARAMETER temperature 0.1\n'
    'PARAMETER num_ctx 4096\n'
    'PARAMETER stop "<|end|>"\n'
)
with open(f'{SAVE_DIR}/gguf/Modelfile', 'w') as f:
    f.write(modelfile)
print('Modelfile written.')

In [ ]:
from huggingface_hub import HfApi, login

login(token='YOUR_HF_TOKEN')  # huggingface.co → Settings → Access Tokens (Write)

api = HfApi()
REPO_ID = 'your-username/truthbite-phi4'  # replace with your HF username

api.create_repo(repo_id=REPO_ID, private=True, exist_ok=True)

api.upload_file(
    path_or_fileobj='/tmp/truthbite-phi4.Q4_K_M.gguf',
    path_in_repo='truthbite-phi4.Q4_K_M.gguf',
    repo_id=REPO_ID,
)
print('GGUF uploaded.')

modelfile_hf = (
    f'FROM hf.co/{REPO_ID}\n\n'
    f'SYSTEM """\n{SYSTEM_PROMPT}\n"""\n\n'
    'TEMPLATE """\n'
    '{{ if .System }}<|system|>\n{{ .System }}<|end|>\n{{ end }}'
    '{{ range .Messages }}{{ if eq .Role "user" }}<|user|>\n{{ .Content }}<|end|>\n'
    '{{ else if eq .Role "assistant" }}<|assistant|>\n{{ .Content }}<|end|>\n'
    '{{ end }}{{ end }}<|assistant|>\n"""\n\n'
    'PARAMETER temperature 0.1\n'
    'PARAMETER num_ctx 4096\n'
    'PARAMETER stop "<|end|>"\n'
)
with open('/tmp/Modelfile', 'w') as f:
    f.write(modelfile_hf)

api.upload_file(
    path_or_fileobj='/tmp/Modelfile',
    path_in_repo='Modelfile',
    repo_id=REPO_ID,
)
print('Modelfile uploaded.')
print('\nTo load into Ollama:')
print('  ollama create truthbite-phi4 -f Modelfile')